# Gerador de Dimensão: Lojas (Centros de Distribuição)

**Objetivo:** criar a carga inicial da dimensão `dim_lojas`, contendo os 6 centros de distribuição fixos (3 no Brasil, 2 no México, 1 na Argentina), incluindo o peso-base de distribuição de vendas de cada um.

**Dependências:** `src/utils/scd2_handler.py` (classe `SCD2Handler`).

**Widget:** `forcar_recriacao` — permite re-executar este notebook durante testes/desenvolvimento.

**Saída:** arquivo JSON gravado na Landing Zone (`/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/lojas/`).

**Observação de modelagem:** esta tabela não referencia diretamente o representante/supervisor responsável, para evitar dependência circular com `dim_representantes` (que referencia o centro ao qual pertence). O relacionamento entre as duas tabelas deve ser obtido via JOIN, no momento da consulta.

In [0]:
# Instalações e configurações iniciais
import sys

sys.path.append("/Workspace/Users/bruno.quelestech@outlook.com/poc-lakehouse-food-latam/src")

from utils.scd2_handler import SCD2Handler

print("Dependências carregadas com sucesso.")

In [0]:
# Widget

dbutils.widgets.dropdown("forcar_recriacao", "false", ["true", "false"])

forcar_recriacao = dbutils.widgets.get("forcar_recriacao") == "true"

print(f"Forçar recriação: {forcar_recriacao}")

In [0]:
# Criação das lojas/CD

lojas_cru = [
    # (loja_id, nome_cidade, pais, peso_distribuicao)
    ("LOJA001", "São Paulo", "Brasil", 0.40),
    ("LOJA002", "Rio de Janeiro", "Brasil", 0.15),
    ("LOJA003", "Minas Gerais", "Brasil", 0.15),
    ("LOJA004", "Guadalajara", "Mexico", 0.125),
    ("LOJA005", "Cidade do México", "Mexico", 0.125),
    ("LOJA006", "Buenos Aires", "Argentina", 0.05),
]

colunas_lojas = ["loja_id", "nome_cidade", "pais", "peso_distribuicao"]

print(f"Total de lojas definidas: {len(lojas_cru)}")
print(f"Soma dos pesos: {sum([l[3] for l in lojas_cru])}")

In [0]:
# Transformação em dataframe spark

df_lojas_cru = spark.createDataFrame(lojas_cru, colunas_lojas)

df_lojas_cru.display()

In [0]:
# Aplicar o SCD2Handler

scd2 = SCD2Handler()
df_lojas_scd2 = scd2.iniciar_controle_scd2(df_lojas_cru)

df_lojas_scd2.display()

In [0]:
# Gravar o resultado como JSON na Landing Zone

caminho_landing = "/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/lojas"

arquivos_existentes = []
try:
    arquivos_existentes = dbutils.fs.ls(caminho_landing)
except Exception:
    pass

if len(arquivos_existentes) > 0 and not forcar_recriacao:
    print("Já existem arquivos na Landing Zone de lojas. "
          "Nenhuma ação foi tomada (use o widget 'forcar_recriacao' = true para sobrescrever).")
else:
    (
        df_lojas_scd2
        .coalesce(1)
        .write
        .mode("overwrite")
        .json(caminho_landing)
    )
    print(f"Arquivo gravado com sucesso em: {caminho_landing}")